# NOTEBOOK 03: EVALUATION DDPG - CODE REEL

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import sys
import torch

sys.path.append('..')
from agent.ddpg import DDPGAgent
from env.inventory_env import InventoryEnv
from evaluate import EvaluationMetrics, BaselinePolicy, EvaluationReporter

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print('Imports OK')

In [ ]:
DATA_PATH = '../data/retail_store_inventory.csv'
MODEL_PATH = '../results/models/ddpg_final'
OUTPUT_DIR = '../results/evaluation'

EVAL_CONFIG = {
    'num_episodes': 20,
    'deterministic': True,
    'seed': 42,
}

COST_CONFIG = {
    'holding_cost_per_unit': 0.5,
    'order_cost_per_order': 10.0,
    'stockout_penalty': 5.0,
    'max_inventory': 1000,
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'OUTPUT_DIR: {OUTPUT_DIR}')

In [ ]:
# Charger l'environnement
env = InventoryEnv(DATA_PATH, normalize=True)
print(f'Env chargee: state_dim={env.state_dim}, action_dim={env.action_dim}')
print(f'Action space: [{env.action_low}, {env.action_high}]')

In [ ]:
# Charger l'agent DDPG
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

agent = DDPGAgent(
    state_dim=env.state_dim,
    action_dim=env.action_dim,
    action_low=env.action_low,
    action_high=env.action_high,
    device=device
)

try:
    agent.load(MODEL_PATH)
    print(f'Agent DDPG charge depuis {MODEL_PATH}')
except FileNotFoundError as e:
    print(f'ERREUR: {e}')
    print(f'Cherchait: {MODEL_PATH}_actor.pth et {MODEL_PATH}_critic.pth')

In [ ]:
# Evaluer DDPG
def evaluate_policy(agent, env, num_episodes, seed=42):
    episodes_data = []
    np.random.seed(seed)
    
    for episode in range(num_episodes):
        state = env.reset(seed=seed + episode)
        
        ep_data = {
            'rewards': [],
            'actions': [],
            'inventory_levels': [],
            'demands': [],
        }
        
        done = False
        
        while not done:
            action = agent.select_action(state, explore=False)
            next_state, reward, done, info = env.step(action)
            
            ep_data['rewards'].append(reward)
            ep_data['actions'].append(float(action) if hasattr(action, '__len__') else action)
            ep_data['inventory_levels'].append(info.get('inventory', 0.0))
            ep_data['demands'].append(info.get('demand', 0.0))
            
            state = next_state
        
        episodes_data.append(ep_data)
        if (episode + 1) % 5 == 0:
            print(f'Episode {episode + 1}: cumulative_reward = {np.sum(ep_data["rewards"]):.2f}')
    
    return episodes_data

episodes_data = evaluate_policy(agent, env, EVAL_CONFIG['num_episodes'], EVAL_CONFIG['seed'])
print(f'Evaluation terminee: {len(episodes_data)} episodes')

In [ ]:
# Calculer les metriques
evaluator = EvaluationMetrics(config=COST_CONFIG)

episodes_metrics = []
for ep_data in episodes_data:
    metrics = evaluator.calculate_episode_cost(
        rewards=ep_data['rewards'],
        actions=ep_data['actions'],
        inventory_levels=ep_data['inventory_levels'],
        demands=ep_data['demands']
    )
    episodes_metrics.append(metrics)

trained_metrics = evaluator.calculate_policy_performance(episodes_metrics)

print('\n' + '='*60)
print('METRIQUES DDPG')
print('='*60)
for key, value in trained_metrics.items():
    if isinstance(value, float):
        print(f'{key:.<40} {value:>15.2f}')

In [ ]:
# Evaluer les baselines
def eval_baseline(baseline_func, env, num_episodes, seed=42):
    episodes_data = []
    np.random.seed(seed)
    
    for episode in range(num_episodes):
        state = env.reset(seed=seed + episode)
        ep_data = {'rewards': [], 'actions': [], 'inventory_levels': [], 'demands': []}
        done = False
        
        while not done:
            action = baseline_func(state)
            next_state, reward, done, info = env.step(action)
            ep_data['rewards'].append(reward)
            ep_data['actions'].append(action)
            ep_data['inventory_levels'].append(info.get('inventory', 0.0))
            ep_data['demands'].append(info.get('demand', 0.0))
            state = next_state
        
        episodes_data.append(ep_data)
    return episodes_data

# Greedy policy
def greedy(state):
    inventory = state[0]
    if inventory < 0.3:
        return 100.0
    return 0.0

greedy_data = eval_baseline(greedy, env, EVAL_CONFIG['num_episodes'], EVAL_CONFIG['seed'])
greedy_metrics_list = [evaluator.calculate_episode_cost(**ep) for ep in [
    {'rewards': ed['rewards'], 'actions': ed['actions'], 
     'inventory_levels': ed['inventory_levels'], 'demands': ed['demands']}
    for ed in greedy_data
]]
greedy_metrics = evaluator.calculate_policy_performance(greedy_metrics_list)

print('\nBaseline Greedy:')
print(f'  Coût moyen: {greedy_metrics["mean_episode_cost"]:.2f}')
print(f'  DDPG vs Greedy: {(greedy_metrics["mean_episode_cost"] - trained_metrics["mean_episode_cost"])/greedy_metrics["mean_episode_cost"]*100:.1f}% reduction')

In [ ]:
# Sauvegarder les resultats
reporter = EvaluationReporter()
reporter.generate_evaluation_report(trained_metrics, greedy_metrics, OUTPUT_DIR)

df_metrics = pd.DataFrame(episodes_metrics)
df_metrics.to_csv(os.path.join(OUTPUT_DIR, 'episodes_metrics.csv'), index=False)
print(f'Metriques sauvegardees')

In [ ]:
# Visualiser l'episode optimal
best_idx = np.argmin([ep['total_cost'] for ep in episodes_metrics])
best_ep = episodes_data[best_idx]
print(f'\nEpisode optimal: #{best_idx}')
print(f'  Coût: {episodes_metrics[best_idx]["total_cost"]:.2f}')
print(f'  Reward: {episodes_metrics[best_idx]["episode_reward"]:.2f}')

reporter.plot_episode_trajectory(
    best_ep['inventory_levels'],
    best_ep['demands'],
    best_ep['actions'],
    best_ep['rewards'],
    title=f'Episode optimal (coût={episodes_metrics[best_idx]["total_cost"]:.2f})'
)

In [ ]:
# Breakdown des couts
reporter.plot_cost_breakdown(episodes_metrics, title='Repartition des couts DDPG')

In [ ]:
# Comparaison
reporter.plot_comparison(trained_metrics, greedy_metrics)